# Demonstração — Jacobiano, redução de parâmetros e crescimento da complexidade

Este notebook foi feito para apresentação.

Objetivos:

1. mostrar o problema \(10\choose4\);
2. mostrar como a complexidade cresce com o número de períodos;
3. mostrar a redução do Jacobiano + SVD de \(30\to17\) parâmetros por período;
4. mostrar que a economia absoluta aumenta com a complexidade;
5. explicar por que o Jacobiano conjunto é block-local no ponto Dicke uniforme;
6. comparar o budget fixo com o número de parâmetros;
7. carregar automaticamente resultados reais do projeto, caso estejam na mesma pasta.

O notebook procura arquivos `.csv` e `.npz` na própria pasta de execução.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
FIG_DIR = ROOT / "figures_apresentacao"
FIG_DIR.mkdir(exist_ok=True)

N = 10
K = 4
M = math.comb(N, K)

LOCAL_PARAMS = 30
R90_FALLBACK = 17
ENERGY_THRESHOLD = 0.90

BUDGET = 128
T_MAX = 8

print("Pasta:", ROOT)
print("Estados factíveis por período =", M)
print("Parâmetros locais FULL =", LOCAL_PARAMS)

## 1. Problema local: escolher 4 entre 10

Para um período:

\[
x\in\{0,1\}^{10},
\qquad
\sum_{i=1}^{10}x_i=4.
\]

Logo:

\[
M=\binom{10}{4}=210.
\]

No ponto Dicke uniforme:

\[
p_x=\frac{1}{210}.
\]

In [ ]:
periods = np.arange(1, T_MAX + 1)

scale_df = pd.DataFrame({
    "periodos": periods,
    "log10_trajetorias": periods * np.log10(M),
})

scale_df["trajetorias_exatas"] = [M**int(t) for t in periods]

display(scale_df)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    scale_df["periodos"],
    scale_df["log10_trajetorias"],
    marker="o"
)

ax.set_xlabel("Número de períodos T")
ax.set_ylabel(r"$\log_{10}$ do número de trajetórias")
ax.set_title(r"Crescimento combinatório: $210^T$")
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(FIG_DIR / "01_crescimento_combinatorio.png", dpi=200)
plt.show()

for t in [1, 2, 4]:
    print(f"T={t}: {M**t:,} trajetórias")

## 2. Jacobiano de probabilidade

Definimos:

\[
J_{xj}
=
\frac{\partial p_x}{\partial\theta_j}.
\]

No circuito local:

\[
J\in\mathbb{R}^{210\times30}.
\]

Como:

\[
\sum_x p_x(\theta)=1,
\]

então:

\[
\sum_x
\frac{\partial p_x}{\partial\theta_j}
=0.
\]

Ou seja:

\[
\boxed{\mathbf{1}^T J=0}.
\]

Isso não significa que cada derivada é zero. Significa que aumentos e diminuições de probabilidade se compensam.

## 3. SVD: de 30 para 17 parâmetros

Fazemos:

\[
J=U\Sigma V^T.
\]

Selecionamos o menor \(r\) tal que:

\[
\frac{\sum_{i=1}^{r}\sigma_i^2}
{\sum_{i=1}^{30}\sigma_i^2}
\ge0.90.
\]

No projeto:

\[
\boxed{r_{90}=17}.
\]

Portanto:

\[
\boxed{30\rightarrow17}
\]

parâmetros efetivos por período.

In [ ]:
def find_local_jacobian_or_svals(folder):
    candidates = []

    for path in sorted(folder.glob("*.npz")):
        try:
            data = np.load(path, allow_pickle=True)
            keys = set(data.files)

            for key in ["J_u", "J", "jacobian", "J_local"]:
                if key in keys:
                    arr = np.asarray(data[key], float)
                    if arr.ndim == 2 and arr.shape[1] == LOCAL_PARAMS:
                        candidates.append(("J", path, key, arr))

            for key in ["singular_values", "svals", "S"]:
                if key in keys:
                    arr = np.asarray(data[key], float).ravel()
                    if len(arr) >= 2:
                        candidates.append(("S", path, key, arr))
        except Exception:
            pass

    if not candidates:
        return None

    candidates.sort(key=lambda x: 0 if x[0] == "J" else 1)
    return candidates[0]


found = find_local_jacobian_or_svals(ROOT)

J_local = None
singular_values = None

if found is not None:
    kind, path, key, arr = found

    if kind == "J":
        J_local = arr
        singular_values = np.linalg.svd(J_local, compute_uv=False)
    else:
        singular_values = arr

    frac = np.cumsum(singular_values**2) / np.sum(singular_values**2)
    R90 = int(np.searchsorted(frac, ENERGY_THRESHOLD) + 1)

    print("Arquivo encontrado:", path.name)
    print("r90 calculado =", R90)

    if J_local is not None:
        err = float(np.max(np.abs(J_local.sum(axis=0))))
        print("max |sum_x J_xj| =", f"{err:.3e}")
else:
    R90 = R90_FALLBACK
    print("Jacobiano não encontrado.")
    print("Usando valor congelado do projeto: r90 =", R90)

REDUCTION_PCT = 100 * (1 - R90 / LOCAL_PARAMS)
print(f"Redução = {REDUCTION_PCT:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))

labels = ["FULL local", "Jacobiano + SVD"]
values = [LOCAL_PARAMS, R90]

bars = ax.bar(labels, values)

ax.set_ylabel("Número de parâmetros")
ax.set_title("Compressão em um único período")

for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.6,
        str(value),
        ha="center"
    )

ax.text(
    0.5,
    max(values) * 0.58,
    f"redução = {REDUCTION_PCT:.1f}%",
    ha="center",
    fontsize=13
)

ax.set_ylim(0, max(values) * 1.18)

fig.tight_layout()
fig.savefig(FIG_DIR / "02_reducao_local.png", dpi=200)
plt.show()

## 4. Aumentando a complexidade

Para \(T\) períodos:

\[
N_{\rm FULL}(T)=30T.
\]

Com Jacobiano + SVD:

\[
N_J(T)=17T.
\]

A economia absoluta é:

\[
N_{\rm economizados}(T)
=
(30-17)T
=
13T.
\]

Exemplos:

\[
T=1:\quad30\rightarrow17,
\]

\[
T=2:\quad60\rightarrow34,
\]

\[
T=4:\quad120\rightarrow68,
\]

\[
T=8:\quad240\rightarrow136.
\]

Enquanto isso, o espaço de trajetórias cresce como:

\[
210^T.
\]

In [ ]:
complexity_df = pd.DataFrame({"periodos": periods})

complexity_df["parametros_FULL"] = LOCAL_PARAMS * complexity_df["periodos"]
complexity_df["parametros_Jacobiano"] = R90 * complexity_df["periodos"]

complexity_df["parametros_economizados"] = (
    complexity_df["parametros_FULL"]
    - complexity_df["parametros_Jacobiano"]
)

complexity_df["reducao_percentual"] = (
    100
    * complexity_df["parametros_economizados"]
    / complexity_df["parametros_FULL"]
)

complexity_df["avaliacoes_por_parametro_FULL"] = (
    BUDGET / complexity_df["parametros_FULL"]
)

complexity_df["avaliacoes_por_parametro_Jacobiano"] = (
    BUDGET / complexity_df["parametros_Jacobiano"]
)

display(complexity_df.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    complexity_df["periodos"],
    complexity_df["parametros_FULL"],
    marker="o",
    label="FULL"
)

ax.plot(
    complexity_df["periodos"],
    complexity_df["parametros_Jacobiano"],
    marker="o",
    label="Jacobiano + SVD"
)

ax.fill_between(
    complexity_df["periodos"],
    complexity_df["parametros_Jacobiano"],
    complexity_df["parametros_FULL"],
    alpha=0.15,
    label="Parâmetros economizados"
)

ax.set_xlabel("Número de períodos T")
ax.set_ylabel("Número de parâmetros")
ax.set_title("FULL vs Jacobiano + SVD")
ax.legend()
ax.grid(alpha=0.25)

for t in [1, 2, 4, 8]:
    row = complexity_df.loc[
        complexity_df["periodos"].eq(t)
    ].iloc[0]

    ax.annotate(
        f"{int(row['parametros_FULL'])}→{int(row['parametros_Jacobiano'])}",
        (row["periodos"], row["parametros_Jacobiano"]),
        xytext=(5, -18),
        textcoords="offset points"
    )

fig.tight_layout()
fig.savefig(FIG_DIR / "03_parametros_vs_complexidade.png", dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.bar(
    complexity_df["periodos"],
    complexity_df["parametros_economizados"]
)

ax.set_xlabel("Número de períodos T")
ax.set_ylabel("Parâmetros eliminados")
ax.set_title("Economia absoluta de parâmetros")

for _, row in complexity_df.iterrows():
    ax.text(
        row["periodos"],
        row["parametros_economizados"] + 1,
        str(int(row["parametros_economizados"])),
        ha="center"
    )

fig.tight_layout()
fig.savefig(FIG_DIR / "04_parametros_economizados.png", dpi=200)
plt.show()

## 5. Budget fixo

Nos benchmarks usamos:

\[
B=128
\]

avaliações do objetivo.

Definimos:

\[
\eta
=
\frac{B}{N_{\rm parâmetros}}.
\]

Para \(T=1\):

\[
\eta_{\rm FULL}
=
\frac{128}{30}
\approx4.27.
\]

Para \(T=4\):

\[
\eta_{\rm FULL}
=
\frac{128}{120}
\approx1.07,
\]

enquanto:

\[
\eta_J
=
\frac{128}{68}
\approx1.88.
\]

Com o mesmo budget, aumentar \(T\) aumenta a pressão dimensional sobre o otimizador.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    complexity_df["periodos"],
    complexity_df["avaliacoes_por_parametro_FULL"],
    marker="o",
    label="FULL"
)

ax.plot(
    complexity_df["periodos"],
    complexity_df["avaliacoes_por_parametro_Jacobiano"],
    marker="o",
    label="Jacobiano + SVD"
)

ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
    label="1 avaliação / parâmetro"
)

ax.set_xlabel("Número de períodos T")
ax.set_ylabel("Budget / parâmetros")
ax.set_title(f"Pressão do budget fixo ({BUDGET} avaliações)")
ax.legend()
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(FIG_DIR / "05_budget_por_parametro.png", dpi=200)
plt.show()

## 6. Por que o Jacobiano conjunto fica block-local?

Para \(T\) períodos, a distribuição conjunta é:

\[
P(x_1,\ldots,x_T)
=
\prod_{t=1}^{T}p_t(x_t).
\]

Para um parâmetro do período \(t\):

\[
\frac{\partial P}
{\partial\theta_{tj}}
=
J_t(x_t,j)
\prod_{u\neq t}p_u(x_u).
\]

O Gram do Jacobiano é:

\[
G=J_{\rm joint}^{T}J_{\rm joint}.
\]

Um termo entre dois períodos diferentes \(t\neq s\) contém:

\[
\left[
\sum_{x_t}
p_t(x_t)J_t(x_t,j)
\right]
\left[
\sum_{x_s}
p_s(x_s)J_s(x_s,\ell)
\right].
\]

No Dicke uniforme:

\[
p_t(x)=\frac1M.
\]

Então:

\[
\sum_x p(x)J(x,j)
=
\frac1M
\underbrace{\sum_xJ(x,j)}_{0}
=
0.
\]

Logo:

\[
\boxed{G_{ts}=0,\qquad t\neq s}.
\]

Assim:

\[
\boxed{
J_{\rm joint}^{T}J_{\rm joint}
=
M^{-(T-1)}
\operatorname{blockdiag}
(J^TJ,\ldots,J^TJ)
}.
\]

O Jacobiano identifica a geometria local de cada período, mas no ponto Dicke uniforme não cria relações entre períodos.

In [ ]:
T_SHOW = 4

if J_local is not None:
    G_local = J_local.T @ J_local
    denom = np.max(np.abs(G_local))
    G_local_plot = (
        G_local / denom
        if denom > 0
        else G_local.copy()
    )

    G_joint_plot = np.kron(
        np.eye(T_SHOW),
        G_local_plot
    )

    title = (
        "Gram conjunto calculado com o Jacobiano local real"
    )
else:
    block = np.ones((6, 6))
    G_joint_plot = np.kron(
        np.eye(T_SHOW),
        block
    )

    title = (
        "Estrutura matemática do Gram conjunto "
        "(esquema block-local)"
    )

fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    np.abs(G_joint_plot),
    aspect="auto"
)

ax.set_title(title)
ax.set_xlabel("Parâmetros")
ax.set_ylabel("Parâmetros")

block_size = G_joint_plot.shape[0] // T_SHOW

for t in range(1, T_SHOW):
    pos = t * block_size - 0.5
    ax.axhline(pos, linewidth=0.8)
    ax.axvline(pos, linewidth=0.8)

fig.colorbar(
    im,
    ax=ax,
    label="magnitude relativa"
)

fig.tight_layout()
fig.savefig(FIG_DIR / "06_gram_block_local.png", dpi=200)
plt.show()

## 7. Resultados reais do benchmark

Agora procuramos automaticamente um CSV que contenha:

- `arm`;
- `normalized_regret`;
- `parent_instance_id`.

Se o arquivo real estiver na pasta, os valores são recalculados.

Se não estiver, usamos somente alguns valores congelados do projeto para gerar um gráfico de apresentação.

In [ ]:
def find_benchmark_csv(folder):
    candidates = []

    for path in sorted(folder.glob("*.csv")):
        try:
            preview = pd.read_csv(path, nrows=10)
            cols = set(preview.columns)

            required = {
                "arm",
                "normalized_regret",
                "parent_instance_id",
            }

            if required.issubset(cols):
                full = pd.read_csv(path)
                candidates.append(
                    (len(full), path, full)
                )
        except Exception:
            pass

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return candidates[0]


benchmark_found = find_benchmark_csv(ROOT)

REFERENCE_MEDIANS = {
    "BLOCK_DICKE_JACOBIAN_SVD90": 0.171716,
    "RANDOM_SUBSPACE_MATCHED": 0.159037,
    "CHANNEL_NONE_LOCAL": 0.207972,
    "CHANNEL_RD_STRONG": 0.171495,
}

if benchmark_found is not None:
    nrows, benchmark_path, benchmark_df = benchmark_found

    print(
        "CSV real encontrado:",
        benchmark_path.name,
        f"({nrows} linhas)"
    )

    parent_arm = (
        benchmark_df.groupby(
            ["parent_instance_id", "arm"],
            as_index=False
        )
        .agg(
            normalized_regret=(
                "normalized_regret",
                "median"
            )
        )
    )

    empirical = (
        parent_arm.groupby(
            "arm",
            as_index=False
        )
        .agg(
            regret_mediano=(
                "normalized_regret",
                "median"
            ),
            n_pais=(
                "parent_instance_id",
                "nunique"
            )
        )
    )
else:
    benchmark_path = None

    empirical = pd.DataFrame({
        "arm": list(REFERENCE_MEDIANS.keys()),
        "regret_mediano": list(REFERENCE_MEDIANS.values()),
        "n_pais": [np.nan] * len(REFERENCE_MEDIANS),
    })

    print(
        "Nenhum CSV compatível encontrado."
    )
    print(
        "Usando valores congelados apenas como referência."
    )

display(
    empirical.sort_values("regret_mediano")
)

In [ ]:
preferred_order = [
    "CHANNEL_NONE_LOCAL",
    "BLOCK_DICKE_JACOBIAN_SVD90",
    "CHANNEL_RD_STRONG",
    "RANDOM_SUBSPACE_MATCHED",
]

label_map = {
    "CHANNEL_NONE_LOCAL": "Sem cross-period\n68 parâmetros",
    "BLOCK_DICKE_JACOBIAN_SVD90": "Jacobiano block\n68 parâmetros",
    "CHANNEL_RD_STRONG": "RD strong\n68 parâmetros",
    "RANDOM_SUBSPACE_MATCHED": "Global random\n68 parâmetros",
}

plot_df = empirical.loc[
    empirical["arm"].isin(preferred_order)
].copy()

plot_df["ordem"] = plot_df["arm"].map(
    {a: i for i, a in enumerate(preferred_order)}
)

plot_df = plot_df.sort_values("ordem")

if len(plot_df) >= 2:
    fig, ax = plt.subplots(figsize=(9, 5.5))

    bars = ax.bar(
        [
            label_map.get(a, a)
            for a in plot_df["arm"]
        ],
        plot_df["regret_mediano"]
    )

    ax.set_ylabel(
        "Normalized regret (menor é melhor)"
    )

    ax.set_title(
        "Benchmark multiperíodo T=4"
    )

    for bar, value in zip(
        bars,
        plot_df["regret_mediano"]
    ):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            value + 0.003,
            f"{value:.3f}",
            ha="center"
        )

    fig.tight_layout()
    fig.savefig(
        FIG_DIR / "07_regret_benchmark_T4.png",
        dpi=200
    )
    plt.show()
else:
    print(
        "Braços insuficientes para o gráfico empírico."
    )

In [ ]:
def bootstrap_median_ci(
    values,
    reps=10000,
    seed=12345
):
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]

    rng = np.random.default_rng(seed)
    n = len(v)
    boot = np.empty(reps, float)

    for i in range(reps):
        sample = v[
            rng.integers(
                0,
                n,
                size=n
            )
        ]
        boot[i] = np.median(sample)

    return (
        float(np.median(v)),
        float(np.quantile(boot, 0.025)),
        float(np.quantile(boot, 0.975)),
    )


if benchmark_found is not None:
    arms_available = set(
        parent_arm["arm"]
    )

    full_candidates = [
        "FULL_MULTI_DICKE",
        "FULL",
    ]

    full_arm = next(
        (
            a for a in full_candidates
            if a in arms_available
        ),
        None
    )

    jac_arm = (
        "BLOCK_DICKE_JACOBIAN_SVD90"
        if "BLOCK_DICKE_JACOBIAN_SVD90"
        in arms_available
        else None
    )

    if (
        full_arm is not None
        and jac_arm is not None
    ):
        pw = (
            parent_arm.loc[
                parent_arm["arm"].isin(
                    [full_arm, jac_arm]
                )
            ]
            .pivot(
                index="parent_instance_id",
                columns="arm",
                values="normalized_regret"
            )
            .dropna()
        )

        gain = (
            pw[full_arm]
            - pw[jac_arm]
        )

        med, lo, hi = bootstrap_median_ci(
            gain
        )

        print(
            "Teste pareado FULL - Jacobiano"
        )
        print("N pais =", len(gain))
        print(
            f"ganho mediano = {med:.6f}"
        )
        print(
            f"IC95% = [{lo:.6f}, {hi:.6f}]"
        )

        if lo > 0:
            print(
                "Jacobiano apresenta menor regret "
                "com IC95% completamente positivo."
            )
        else:
            print(
                "IC95% cruza zero: não afirmar "
                "superioridade estatística."
            )
    else:
        print(
            "CSV real encontrado, mas FULL e "
            "Jacobiano não estão ambos presentes."
        )
else:
    print(
        "Sem CSV real: teste estatístico não executado."
    )

## 8. Mensagem principal

### Espaço combinatório

\[
N_{\rm trajetórias}(T)
=
210^T.
\]

### Dimensão contínua

Sem redução:

\[
N_{\rm FULL}=30T.
\]

Com Jacobiano + SVD:

\[
N_J=17T.
\]

Logo:

\[
\boxed{
N_{\rm economizados}=13T
}
\]

e:

\[
\boxed{
\text{redução relativa}=43.3\%
}.
\]

### Interpretação

O estado Dicke já restringe o problema ao setor factível.

O Jacobiano revela uma segunda compressibilidade:

\[
30\rightarrow17
\]

direções relevantes por período.

Conforme o número de períodos aumenta, a porcentagem de redução permanece constante, mas a economia absoluta cresce:

\[
13,\ 26,\ 52,\ 78,\ 104,\ldots
\]

parâmetros eliminados.

Ao mesmo tempo, o número de trajetórias cresce exponencialmente como \(210^T\).

### Frase para apresentação

> O Dicke reduz o espaço de soluções; o Jacobiano reduz o espaço de movimentos relevantes. A redução de 30 para 17 parâmetros por período elimina 43.3% das variáveis contínuas. Quando aumentamos o número de períodos, essa economia cresce linearmente, enquanto o espaço combinatório cresce exponencialmente.

In [ ]:
slide_table = complexity_df.loc[
    complexity_df["periodos"].isin(
        [1, 2, 4, 8]
    ),
    [
        "periodos",
        "parametros_FULL",
        "parametros_Jacobiano",
        "parametros_economizados",
        "avaliacoes_por_parametro_FULL",
        "avaliacoes_por_parametro_Jacobiano",
    ]
].copy()

slide_table["trajetorias"] = [
    M**int(t)
    for t in slide_table["periodos"]
]

slide_table = slide_table[
    [
        "periodos",
        "trajetorias",
        "parametros_FULL",
        "parametros_Jacobiano",
        "parametros_economizados",
        "avaliacoes_por_parametro_FULL",
        "avaliacoes_por_parametro_Jacobiano",
    ]
]

display(
    slide_table.round(3)
)

slide_table.to_csv(
    ROOT / "tabela_resumo_apresentacao.csv",
    index=False
)

print("Gráficos:", FIG_DIR)
print(
    "Tabela:",
    ROOT / "tabela_resumo_apresentacao.csv"
)